In [32]:
import pandas as pd
import numpy as np
import joblib # to load the model
import pickle # to load the training features
import matplotlib.pyplot as plt

In [33]:
import warnings
warnings.filterwarnings("ignore")

In [34]:
# Load applicants and check variables
df = pd.read_csv("rejected_applicants.csv")

df.head()

,Applicant_ID,Age,Credit_History_Length,Requested_Loan_Amount,Declared_Income,Co_Applicant_Income,Loan_Term_Months,Past_Bankruptcies
0,0,41,8,19325,64022,0,12,1
1,1,31,16,44181,47038,60000,24,1
2,2,65,10,27058,75310,60000,60,1
3,3,31,4,25092,49420,0,36,1
4,4,30,1,43459,71353,20000,48,1


In [35]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Applicant_ID           50 non-null     int64
 1   Age                    50 non-null     int64
 2   Credit_History_Length  50 non-null     int64
 3   Requested_Loan_Amount  50 non-null     int64
 4   Declared_Income        50 non-null     int64
 5   Co_Applicant_Income    50 non-null     int64
 6   Loan_Term_Months       50 non-null     int64
 7   Past_Bankruptcies      50 non-null     int64
dtypes: int64(8)
memory usage: 3.3 KB


,Applicant_ID,Age,Credit_History_Length,Requested_Loan_Amount,Declared_Income,Co_Applicant_Income,Loan_Term_Months,Past_Bankruptcies
count,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,24.50000,43.680000,8.940000,30650.880000,62890.180000,15200.000000,35.280000,0.940000
std,14.57738,14.563639,5.411288,11900.631542,23174.398883,22336.937469,16.515966,0.239898
min,0.00000,23.000000,1.000000,5510.000000,30792.000000,0.000000,12.000000,0.000000
25%,12.25000,31.000000,4.000000,20570.750000,44823.000000,0.000000,24.000000,1.000000
50%,24.50000,41.000000,9.500000,31534.500000,58785.000000,0.000000,36.000000,1.000000
75%,36.75000,56.750000,13.000000,41427.000000,75193.750000,20000.000000,48.000000,1.000000
max,49.00000,69.000000,19.000000,48548.000000,117851.000000,60000.000000,60.000000,1.000000


In [36]:
# Split mutable and immutable features
immutable_features = [
    "Age",
    "Credit_History_Length",
    "Past_Bankruptcies"
]

mutable_features = [
    "Requested_Loan_Amount",
    "Declared_Income",
    "Co_Applicant_Income",
    "Loan_Term_Months"
]

In [37]:
# load model
model = joblib.load("loan_approval_model.pkl")

print(type(model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [45]:
with open("loan_approval_model.pkl", "rb") as f:
    feature_names = pickle.load(f)

print(feature_names)

['Age' 'Credit_History_Length' 'Requested_Loan_Amount' 'Declared_Income'
 'Co_Applicant_Income' 'Loan_Term_Months' 'Past_Bankruptcies']


In [46]:
X = df[list(feature_names)].copy() # get the model with the trained features 

In [43]:
model.predict(X.head())
model.predict_proba(X.head())

array([[0.92705775, 0.07294225],
       [0.87505811, 0.12494189],
       [0.88065296, 0.11934704],
       [0.91797771, 0.08202229],
       [0.9022937 , 0.0977063 ]])

In [ ]:
# First we need to pick the applicant closer to being accepted so we can modify their data

In [51]:
df["approval_prob"] = model.predict_proba(X)[:,1] # We get only the column 1, the probability of approval
# print(df["approval_prob"])

In [53]:
df.sort_values("approval_prob", ascending=False).head(10) # Get a table sorted by approval rating in order to pick the best applicant

,Applicant_ID,Age,Credit_History_Length,Requested_Loan_Amount,Declared_Income,Co_Applicant_Income,Loan_Term_Months,Past_Bankruptcies,approval_prob
13,13,46,10,33064,40663,0,12,0,0.245652
33,33,67,15,5510,37593,20000,60,1,0.173880
48,48,56,3,41913,34858,40000,12,0,0.168014
45,45,57,4,34894,68733,60000,36,1,0.139926
8,8,52,19,11476,55968,0,36,1,0.127839
1,1,31,16,44181,47038,60000,24,1,0.124942
2,2,65,10,27058,75310,60000,60,1,0.119347
15,15,35,1,48371,40872,20000,12,0,0.115404
14,14,36,15,23676,49053,40000,24,1,0.113988
35,35,26,2,29496,54067,60000,24,1,0.112139


In [55]:
original_applicant = df.sort_values("approval_prob", ascending=False).iloc[[0]].copy()

In [59]:
df["prediction"] = model.predict(X)

In [61]:
df[df["prediction"] == 0].sort_values("approval_prob", ascending=False).head(10)

,Applicant_ID,Age,Credit_History_Length,Requested_Loan_Amount,Declared_Income,Co_Applicant_Income,Loan_Term_Months,Past_Bankruptcies,approval_prob,prediction
13,13,46,10,33064,40663,0,12,0,0.245652,0
33,33,67,15,5510,37593,20000,60,1,0.173880,0
48,48,56,3,41913,34858,40000,12,0,0.168014,0
45,45,57,4,34894,68733,60000,36,1,0.139926,0
8,8,52,19,11476,55968,0,36,1,0.127839,0
1,1,31,16,44181,47038,60000,24,1,0.124942,0
2,2,65,10,27058,75310,60000,60,1,0.119347,0
15,15,35,1,48371,40872,20000,12,0,0.115404,0
14,14,36,15,23676,49053,40000,24,1,0.113988,0
35,35,26,2,29496,54067,60000,24,1,0.112139,0


In [ ]:
# We found the best rejected applicant to work with
# We need to find which features are more important
# We need to permutate these features for the applicant to be accepted and get the least cost score